# SimCLR Baseline Screening Run
**Stage I — Screening protocol:** 200 epochs, 1 seed, CIFAR-10 then STL-10

This notebook runs the full baseline pipeline:
1. Environment check
2. Smoke test (5 epochs) — verify nothing crashes
3. CIFAR-10 pretraining (200 epochs)
4. CIFAR-10 linear evaluation
5. STL-10 pretraining (200 epochs)
6. STL-10 linear evaluation
7. Results summary

Expected runtime: ~1h CIFAR-10 + ~3h STL-10 on T4 GPU.

## 1. Environment Check

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory      : {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

## 2. Clone Repository

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/SAlaMusa/IDL.git"
REPO_DIR = "/kaggle/working/IDL"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print("Repository contents:")
print(os.listdir("."))

## 3. Verify Imports

In [ ]:
# Verify all project modules import cleanly before starting any training
import sys
sys.path.insert(0, REPO_DIR)

from data_aug.contrastive_learning_dataset import ContrastiveLearningDataset
from models.resnet_simclr import ResNetSimCLR
from optimizers.lars import LARS
from simclr import SimCLR

print("All imports OK")

# Quick sanity check: build the model and confirm parameter count
model = ResNetSimCLR(base_model='resnet18', out_dim=128, cifar_stem=True)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"ResNet-18 + projection head: {n_params:.2f}M parameters")
del model

## 4. Helper — Find Latest Checkpoint
After training, `simclr.py` saves checkpoints under `runs/<timestamp>/`.
This helper finds the most recently saved one.

In [ ]:
import glob

def find_latest_checkpoint(run_dir="runs"):
    """Return the path of the most recently modified .pth.tar checkpoint."""
    checkpoints = glob.glob(os.path.join(run_dir, "**", "*.pth.tar"), recursive=True)
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint found under {run_dir}/")
    latest = max(checkpoints, key=os.path.getmtime)
    print(f"Found checkpoint: {latest}")
    return latest

def copy_checkpoint(src, name):
    """Copy a checkpoint to /kaggle/working/ with a human-readable name."""
    import shutil
    dst = f"/kaggle/working/{name}"
    shutil.copy2(src, dst)
    print(f"Saved to: {dst}")
    return dst

## 5. Smoke Test — CIFAR-10, 5 Epochs
Runs a tiny training job to confirm the full pipeline works end-to-end.
Expected: no crashes, loss printed, checkpoint saved. Takes ~3 minutes.

In [ ]:
print("=" * 60)
print("SMOKE TEST — CIFAR-10, 5 epochs, batch 128")
print("=" * 60)

result = subprocess.run(
    [
        "python", "run.py",
        "--config", "configs/baseline_cifar10.yaml",
        "--epochs", "5",
        "--batch-size", "128",
        "-j", "2",
        "--seed", "42",
    ],
    capture_output=False   # stream output directly to notebook
)

if result.returncode != 0:
    raise RuntimeError("Smoke test failed. Fix the error above before continuing.")
print("\nSmoke test PASSED.")

## 6. CIFAR-10 Pretraining — 200 Epochs (Screening)
- ResNet-18 with 3×3 CIFAR stem
- LARS optimizer, lr = 0.3 × 256/256 = 0.3
- Cosine decay with 10-epoch linear warmup
- Expected runtime: ~1 hour on T4

In [ ]:
print("=" * 60)
print("CIFAR-10 SCREENING PRETRAIN — 200 epochs, batch 256")
print("=" * 60)

result = subprocess.run(
    [
        "python", "run.py",
        "--config", "configs/baseline_cifar10.yaml",
        "--epochs", "200",
        "--batch-size", "256",
        "-j", "2",
        "--seed", "42",
    ],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError("CIFAR-10 pretraining failed.")

cifar10_ckpt = find_latest_checkpoint()
cifar10_ckpt_saved = copy_checkpoint(cifar10_ckpt, "cifar10_resnet18_ep200_seed42.pth.tar")
print("CIFAR-10 pretraining complete.")

## 7. CIFAR-10 Linear Evaluation
Freezes the pretrained encoder and trains a linear classifier for 100 epochs.
SGD momentum 0.9, lr=0.1 with cosine decay. Expected runtime: ~10 minutes.

In [ ]:
print("=" * 60)
print("CIFAR-10 LINEAR EVALUATION")
print("=" * 60)

result = subprocess.run(
    [
        "python", "linear_eval.py",
        "--checkpoint", cifar10_ckpt_saved,
        "--dataset", "cifar10",
        "--arch", "resnet18",
        "--epochs", "100",
        "-b", "256",
        "-j", "2",
        "--seed", "42",
    ],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError("CIFAR-10 linear eval failed.")
print("CIFAR-10 linear evaluation complete.")

## 8. STL-10 Pretraining — 200 Epochs (Screening)
- ResNet-18, standard 7×7 stem (STL-10 images are 96×96)
- LARS optimizer, lr = 0.3 × 128/256 = 0.15
- Pretrains on the 100k unlabeled split
- Expected runtime: ~3 hours on T4

> **Note:** If the Kaggle session is running low on time after CIFAR-10,
> save your work and continue STL-10 in a new session using the saved CIFAR-10 checkpoint.

In [ ]:
print("=" * 60)
print("STL-10 SCREENING PRETRAIN — 200 epochs, batch 128")
print("=" * 60)

result = subprocess.run(
    [
        "python", "run.py",
        "--config", "configs/baseline_stl10.yaml",
        "--epochs", "200",
        "--batch-size", "128",
        "-j", "2",
        "--seed", "42",
    ],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError("STL-10 pretraining failed.")

stl10_ckpt = find_latest_checkpoint()
stl10_ckpt_saved = copy_checkpoint(stl10_ckpt, "stl10_resnet18_ep200_seed42.pth.tar")
print("STL-10 pretraining complete.")

## 9. STL-10 Linear Evaluation
Linear eval uses the labeled train split (5,000 images) and test split.
Expected runtime: ~5 minutes.

In [ ]:
print("=" * 60)
print("STL-10 LINEAR EVALUATION")
print("=" * 60)

result = subprocess.run(
    [
        "python", "linear_eval.py",
        "--checkpoint", stl10_ckpt_saved,
        "--dataset", "stl10",
        "--arch", "resnet18",
        "--epochs", "100",
        "-b", "256",
        "-j", "2",
        "--seed", "42",
    ],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError("STL-10 linear eval failed.")
print("STL-10 linear evaluation complete.")

## 10. Results Summary
List saved checkpoints and print a summary table for recording results.

In [ ]:
import os

print("=" * 60)
print("SAVED OUTPUTS IN /kaggle/working/")
print("=" * 60)
for f in sorted(os.listdir("/kaggle/working/")):
    path = os.path.join("/kaggle/working/", f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f:<50}  {size_mb:.1f} MB")

print()
print("=" * 60)
print("RECORD THESE IN YOUR RESULTS CSV")
print("=" * 60)
print(f"{'run_name':<35} {'dataset':<10} {'arch':<10} {'epochs':<8} {'batch':<8} {'seed'}")
print("-" * 80)
print(f"{'baseline_screen':<35} {'cifar10':<10} {'resnet18':<10} {'200':<8} {'256':<8} {'42'}")
print(f"{'baseline_screen':<35} {'stl10':<10} {'resnet18':<10} {'200':<8} {'128':<8} {'42'}")
print()
print("Fill in linear_eval_acc from the 'Best Test Top-1 Accuracy' lines printed above.")